# Step 02 — Feature Engineering

This notebook explores the engineered feature matrix produced by `pipelines/02_features.py`.
Features are derived from the raw macro series via cross-asset ratios, log transforms,
Bernstein polynomial gap-filling, and smoothed first/second/third derivatives.

**Prerequisites:** Run `python pipelines/02_features.py` first (or `python run_pipeline.py --steps 1,2`).

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import plotting, DATA_DIR

setup_logging()
log = logging.getLogger(__name__)

cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("Config loaded")
print("RunConfig:", run_cfg)

## Load Feature Matrix

The processed feature matrix is stored at `data/processed/features.parquet`.
Each row is a quarter; columns include raw series, log-transforms, and derivatives.

In [ ]:
features_path = DATA_DIR / "processed" / "features.parquet"

try:
    features = pd.read_parquet(features_path)
    print(f"Loaded features: {features.shape[0]} quarters × {features.shape[1]} columns")
    print(f"Date range: {features.index.min()} — {features.index.max()}")
except FileNotFoundError:
    print(f"ERROR: {features_path} not found.")
    print("Run `python pipelines/02_features.py` first to generate the feature file.")
    features = None

## Column Groups

Features are organized by prefix:
- **Raw / derived**: `sp500`, `us_infl`, `credit_spread`, etc.
- **Log-transformed**: columns beginning with `log_`
- **First derivative**: columns ending with `_d1`
- **Second derivative**: columns ending with `_d2`
- **Third derivative**: columns ending with `_d3`

In [ ]:
if features is not None:
    cols = features.columns.tolist()

    log_cols  = [c for c in cols if c.startswith("log_")]
    d1_cols   = [c for c in cols if c.endswith("_d1")]
    d2_cols   = [c for c in cols if c.endswith("_d2")]
    d3_cols   = [c for c in cols if c.endswith("_d3")]
    raw_cols  = [c for c in cols if c not in log_cols + d1_cols + d2_cols + d3_cols]

    print(f"Raw / derived columns  : {len(raw_cols):3d}  — {raw_cols[:5]} ...")
    print(f"Log-transformed (log_) : {len(log_cols):3d}  — {log_cols[:5]} ...")
    print(f"First derivative (_d1) : {len(d1_cols):3d}  — {d1_cols[:5]} ...")
    print(f"Second derivative (_d2): {len(d2_cols):3d}  — {d2_cols[:5]} ...")
    print(f"Third derivative (_d3) : {len(d3_cols):3d}  — {d3_cols[:5]} ...")
    print(f"Total                  : {len(cols):3d}")

## Feature Distributions

Histograms for a subset of features, excluding higher-order derivatives.
Useful for detecting outliers or skewed distributions.

In [ ]:
if features is not None:
    plotting.plot_feature_distributions(features, run_cfg)

## Feature Correlations

Correlation heatmap for the top-40 most-variable features.
Highly correlated feature groups confirm expected economic relationships
(e.g., equity valuation metrics move together).

In [ ]:
if features is not None:
    plotting.plot_feature_correlations(features, run_cfg, top_n=40)

## NaN Counts per Column

After gap-filling, most interior NaNs should be resolved.
Remaining NaNs indicate periods outside the data range of a given series.

In [ ]:
if features is not None:
    nan_counts = features.isna().sum().sort_values(ascending=False)
    non_zero = nan_counts[nan_counts > 0]
    if non_zero.empty:
        print("No NaNs in feature matrix — gap filling was complete.")
    else:
        print(f"Columns with NaN values ({len(non_zero)} of {len(features.columns)}):")
        print(non_zero.to_string())

## Describe Statistics (top 20 by std)

Summary statistics sorted by standard deviation — the most variable features
are often the most informative for clustering.

In [ ]:
if features is not None:
    desc = features.describe().T.sort_values("std", ascending=False)
    print("Top 20 features by standard deviation:")
    display(desc.head(20))